In [1]:
# Fine tune modernbert base embed model with squad data to create a fine tuned embedding model
# which should be better than sentence transformer model
import pandas as pd
train_df = pd.read_csv("../data/squad_v2/train.csv")
val_df = pd.read_csv("../data/squad_v2/validation.csv")
len(train_df), len(val_df)

(130319, 11873)

In [3]:
# Sample N records
train_df.head(100)

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': array(['in the late 1990s'], dtype=ob..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': array(['singing and dancing'], dtype=..."
2,56be85543aeaaa14008c9066,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,"{'text': array(['2003'], dtype=object), 'answe..."
3,56bf6b0f3aeaaa14008c9601,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,"{'text': array(['Houston, Texas'], dtype=objec..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,"{'text': array(['late 1990s'], dtype=object), ..."
...,...,...,...,...,...
95,56d462f82ccc5a1400d8311f,Beyoncé,LeToya Luckett and Roberson became unhappy wit...,Who replaced Luckett and Roberson in Destiny's...,{'text': array(['Farrah Franklin and Michelle ...
96,56d462f82ccc5a1400d83120,Beyoncé,LeToya Luckett and Roberson became unhappy wit...,Who was blamed for Luckett and Roberson leavin...,"{'text': array(['Beyoncé'], dtype=object), 'an..."
97,56d462f82ccc5a1400d83121,Beyoncé,LeToya Luckett and Roberson became unhappy wit...,Who helped Beyoncé overcome her depression dur...,"{'text': array(['her mother'], dtype=object), ..."
98,56d462f82ccc5a1400d83123,Beyoncé,LeToya Luckett and Roberson became unhappy wit...,Which newest member was removed from Destiny's...,"{'text': array(['Farrah Franklin'], dtype=obje..."


In [8]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[["question", "context"]].head(100).rename(columns={'question': 'query'}), preserve_index=False)
test_dataset =  Dataset.from_pandas(val_df[["question", "context"]].head(100).rename(columns={'question': 'query'}), preserve_index=False)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
train_dataset


Train dataset size: 100
Test dataset size: 100


Dataset({
    features: ['query', 'context'],
    num_rows: 100
})

In [9]:
# Fine Tuning 

import torch
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformer

model_name = "../model/modernbert-embed-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)

loss = MultipleNegativesRankingLoss(model)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 213.83it/s]


In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments , SentenceTransformerTrainer

args = SentenceTransformerTrainingArguments(
    output_dir="../output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=False, # i want to use fp32
    eval_strategy="steps",
    eval_steps=0.125,
    save_strategy="steps",
    save_steps=0,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # tokenizer=model,
    loss=loss,
)

trainer.train()

H:\pythonProject\machine_learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
7,0.693148,0.245522
14,0.361213,0.204680
21,0.188303,0.193386
28,0.594038,0.187694
35,1.093429,0.185478
